# Customer Intelligence — Análise Exploratória de Dados

## Objetivo

Analisar o conjunto de dados Online Retail, disponibilizado pela UCI Machine Learning Repository, para compreender a estrutura das transações comerciais, identificar possíveis problemas de qualidade e investigar padrões iniciais de comportamento dos clientes.

Esta etapa servirá como base para a futura segmentação de clientes utilizando o modelo RFM e o algoritmo K-Means.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

In [2]:
print(sys.executable)

c:\Users\pedro\OneDrive\projetos\customer-intelligence-ml\.venv\Scripts\python.exe


In [3]:
# Caminho do arquivo
file_path = "../data/raw/Online Retail.xlsx"

# Carregando os dados
df = pd.read_excel(file_path)

# Exibindo as primeiras linhas
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [4]:
print(f"Quantidade de registros: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

Quantidade de registros: 541909
Quantidade de colunas: 8


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [6]:
# Quantidade de valores ausentes em cada coluna
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
# Percentual de valores ausentes em cada coluna
(df.isnull().mean() * 100).round(2)

InvoiceNo       0.00
StockCode       0.00
Description     0.27
Quantity        0.00
InvoiceDate     0.00
UnitPrice       0.00
CustomerID     24.93
Country         0.00
dtype: float64

In [8]:
# Quantidade de linhas completamente duplicadas
df.duplicated().sum()


np.int64(5268)

In [9]:
df[df.duplicated()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
587,536412,22273,FELTCRAFT DOLL MOLLY,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
589,536412,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,1,2010-12-01 11:49:00,3.75,17920.0,United Kingdom
594,536412,22141,CHRISTMAS CRAFT TREE TOP ANGEL,1,2010-12-01 11:49:00,2.10,17920.0,United Kingdom
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
600,536412,22569,FELTCRAFT CUSHION BUTTERFLY,2,2010-12-01 11:49:00,3.75,17920.0,United Kingdom


In [10]:
# Identificar as linhas duplicadas
duplicados = df[df.duplicated(keep=False)]

# Mostrar registros da primeira duplicação encontrada
duplicados[
    (duplicados["InvoiceNo"] == duplicados.iloc[0]["InvoiceNo"]) &
    (duplicados["StockCode"] == duplicados.iloc[0]["StockCode"])
]


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom


In [11]:
# Verificar a quantidade de registros com Quantity negativa
(df["Quantity"] < 0).sum()

np.int64(10624)

In [12]:
# Visualizar alguns registros com quantidade negativa
df[df["Quantity"] < 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [13]:
# Analisar registros com preço igual ou inferior a zero
registros_preco_invalido = df[df["UnitPrice"] <= 0]

print(f"Quantidade de registros com preço igual ou inferior a zero: {len(registros_preco_invalido)}")

display(registros_preco_invalido.head(10))

Quantidade de registros com preço igual ou inferior a zero: 2517


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom


In [14]:
# Quantidade de registros com preço zero
print(f"Quantidade de registros com preço igual a zero: {(df['UnitPrice'] == 0).sum()}")

# Quantidade de registros com preço negativo
print(f"Quantidade de registros com preço negativo: {(df['UnitPrice'] < 0).sum()}")


# Visualizar registros com preço negativo
df[df["UnitPrice"] < 0]

Quantidade de registros com preço igual a zero: 2515
Quantidade de registros com preço negativo: 2


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


In [15]:
# Visualizar os registros com preço zero
df[df["UnitPrice"] == 0][
    ["InvoiceNo", "StockCode", "Description",
     "Quantity", "CustomerID", "Country"]
].head(10)

,InvoiceNo,StockCode,Description,Quantity,CustomerID,Country
622,536414,22139,NaN,56,NaN,United Kingdom
1970,536545,21134,NaN,1,NaN,United Kingdom
1971,536546,22145,NaN,1,NaN,United Kingdom
1972,536547,37509,NaN,1,NaN,United Kingdom
1987,536549,85226A,NaN,1,NaN,United Kingdom
1988,536550,85044,NaN,1,NaN,United Kingdom
2024,536552,20950,NaN,1,NaN,United Kingdom
2025,536553,37461,NaN,3,NaN,United Kingdom
2026,536554,84670,NaN,23,NaN,United Kingdom
2406,536589,21777,NaN,-10,NaN,United Kingdom


In [16]:
# Identificar as descrições mais frequentes entre os registros com preço zero
df.loc[df["UnitPrice"] == 0, "Description"].value_counts().head(15)

Description
check                            159
?                                 47
damages                           45
damaged                           43
found                             25
sold as set on dotcom             20
adjustment                        16
Damaged                           14
FRENCH BLUE METAL DOOR SIGN 1      9
thrown away                        9
Unsaleable, destroyed.             9
amazon                             8
FRENCH BLUE METAL DOOR SIGN 8      8
Found                              8
FRENCH BLUE METAL DOOR SIGN 4      7
Name: count, dtype: int64

In [17]:
# Registros com preço zero e CustomerID preenchido
print(f"Quantidade de registros com preço zero e CustomerID preenchido: {df[(df['UnitPrice'] == 0) & (df['CustomerID'].notna())].shape[0]}")

# Registros com preço zero e CustomerID ausente
print(f"Quantidade de registros com preço zero e CustomerID ausente: {df[(df['UnitPrice'] == 0) & (df['CustomerID'].isna())].shape[0]}")

Quantidade de registros com preço zero e CustomerID preenchido: 40
Quantidade de registros com preço zero e CustomerID ausente: 2475


In [18]:
# Criar uma cópia do dataset original para realizar o tratamento
df_clean = df.copy()

# Verificar as dimensões da cópia
print("Dataset original:", df.shape)
print("Dataset tratado:", df_clean.shape)

Dataset original: (541909, 8)
Dataset tratado: (541909, 8)


In [19]:
# Contar registros sem identificação de cliente
df_clean["CustomerID"].isna().sum()

np.int64(135080)

In [20]:
# Remover registros sem identificação de cliente
df_clean = df_clean.dropna(subset=["CustomerID"])

In [21]:
print("Registros após a remoção:", df_clean.shape[0])
print("Clientes sem identificação:", df_clean["CustomerID"].isna().sum())

Registros após a remoção: 406829
Clientes sem identificação: 0


In [22]:
# Visualizar registros com preço zero e cliente identificado
df_clean[df_clean["UnitPrice"] == 0][
    ["InvoiceNo", "StockCode", "Description",
     "Quantity", "CustomerID", "Country", "UnitPrice"]
]

,InvoiceNo,StockCode,Description,Quantity,CustomerID,Country,UnitPrice
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,12647.0,Germany,0.0
33576,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,16560.0,United Kingdom,0.0
40089,539722,22423,REGENCY CAKESTAND 3 TIER,10,14911.0,EIRE,0.0
47068,540372,22090,PAPER BUNTING RETROSPOT,24,13081.0,United Kingdom,0.0
47070,540372,22553,PLASTERS IN TIN SKULLS,24,13081.0,United Kingdom,0.0
56674,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,15107.0,United Kingdom,0.0
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,17560.0,United Kingdom,0.0
130188,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,13239.0,United Kingdom,0.0
139453,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,13113.0,United Kingdom,0.0
145208,548871,22162,HEART GARLAND RUSTIC PADDED,2,14410.0,United Kingdom,0.0


In [23]:
# Contar quantos registros com preço zero ainda existem
(df_clean["UnitPrice"] == 0).sum()

np.int64(40)

In [24]:
# Identificar as faturas que possuem produtos com preço zero
faturas_preco_zero = df_clean.loc[
    df_clean["UnitPrice"] == 0, "InvoiceNo"
].unique()

# Visualizar todas as linhas dessas faturas
df_clean[df_clean["InvoiceNo"].isin(faturas_preco_zero)].sort_values(
    by="InvoiceNo"
)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
9287,537197,22843,BISCUIT TIN VINTAGE GREEN,2,2010-12-05 14:02:00,6.75,12647.0,Germany
9303,537197,22847,BREAD BIN DINER STYLE IVORY,1,2010-12-05 14:02:00,16.95,12647.0,Germany
9302,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,2010-12-05 14:02:00,0.00,12647.0,Germany
9301,537197,22624,IVORY KITCHEN SCALES,2,2010-12-05 14:02:00,8.50,12647.0,Germany
9299,537197,84968A,SET OF 16 VINTAGE ROSE CUTLERY,1,2010-12-05 14:02:00,12.75,12647.0,Germany
...,...,...,...,...,...,...,...,...
485985,577696,M,Manual,1,2011-11-21 11:57:00,0.00,16406.0,United Kingdom
485986,577696,16216,LETTER SHAPE PENCIL SHARPENER,5,2011-11-21 11:57:00,0.06,16406.0,United Kingdom
485987,577696,23368,SET 12 COLOUR PENCILS DOLLY GIRL,2,2011-11-21 11:57:00,0.65,16406.0,United Kingdom
485989,577696,23184,BULL DOG BOTTLE OPENER,1,2011-11-21 11:57:00,4.95,16406.0,United Kingdom


In [25]:
# Criar uma coluna temporária com o valor de cada linha
df_temp = df_clean.copy()
df_temp["TotalLinha"] = df_temp["Quantity"] * df_temp["UnitPrice"]

# Resumir as faturas
resumo_faturas = (
    df_temp[df_temp["InvoiceNo"].isin(faturas_preco_zero)]
    .groupby("InvoiceNo")
    .agg(
        quantidade_itens=("StockCode", "count"),
        itens_pagos=("UnitPrice", lambda x: (x > 0).sum()),
        valor_total=("TotalLinha", "sum")
    )
)

resumo_faturas.head(15)

,quantidade_itens,itens_pagos,valor_total
InvoiceNo,,,
537197,17,16,286.50
539263,21,20,332.18
539722,29,28,762.30
540372,171,169,3527.64
541109,2,1,14.95
543599,1,0,0.00
547417,22,21,329.56
548318,7,6,453.05
548871,7,6,91.48


In [26]:
# Identificar faturas sem itens com preço positivo
resumo_faturas[resumo_faturas["itens_pagos"] == 0]

,quantidade_itens,itens_pagos,valor_total
InvoiceNo,,,
543599,1,0,0.0
564651,4,0,0.0
568384,1,0,0.0
578841,1,0,0.0


In [27]:
invoices = ["543599", "564651", "568384", "578841"]

df_clean[df_clean["InvoiceNo"].astype(str).isin(invoices)]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
86789,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,2011-02-10 13:08:00,0.0,17560.0,United Kingdom
314745,564651,23270,SET OF 2 CERAMIC PAINTED HEARTS,96,2011-08-26 14:19:00,0.0,14646.0,Netherlands
314746,564651,23268,SET OF 2 CERAMIC CHRISTMAS REINDEER,192,2011-08-26 14:19:00,0.0,14646.0,Netherlands
314747,564651,22955,36 FOIL STAR CAKE CASES,144,2011-08-26 14:19:00,0.0,14646.0,Netherlands
314748,564651,21786,POLKADOT RAIN HAT,144,2011-08-26 14:19:00,0.0,14646.0,Netherlands
361825,568384,M,Manual,1,2011-09-27 09:46:00,0.0,12748.0,United Kingdom
502122,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.0,13256.0,United Kingdom


In [28]:
# Identificar faturas de cancelamento
cancelamentos = df_clean["InvoiceNo"].astype(str).str.startswith("C")

# Comparar os registros
print("Total de registros:", len(df_clean))

print("Registros com quantidade negativa:",
      (df_clean["Quantity"] < 0).sum())

print("Registros com fatura iniciada em C:",
      cancelamentos.sum())

print("Registros negativos em faturas iniciadas em C:",
      ((df_clean["Quantity"] < 0) & cancelamentos).sum())

Total de registros: 406829
Registros com quantidade negativa: 8905
Registros com fatura iniciada em C: 8905
Registros negativos em faturas iniciadas em C: 8905


In [29]:
df_clean[
    df_clean["InvoiceNo"].astype(str).str.startswith("C")
].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [30]:
# Filtrar registros de cancelamento
df_cancelamentos = df_clean[
    df_clean["InvoiceNo"].astype(str).str.startswith("C")
].copy()

# Analisar os cancelamentos
print("Total de cancelamentos:", len(df_cancelamentos))

print(
    "Clientes sem identificação:",
    df_cancelamentos["CustomerID"].isna().sum()
)

print(
    "Clientes distintos:",
    df_cancelamentos["CustomerID"].nunique()
)

print(
    "Quantidade total de itens cancelados:",
    df_cancelamentos["Quantity"].sum()
)

display(df_cancelamentos.head(10))

Total de cancelamentos: 8905
Clientes sem identificação: 0
Clientes distintos: 1589
Quantidade total de itens cancelados: -274808


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,2010-12-01 10:24:00,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,2010-12-01 12:38:00,4.25,17897.0,United Kingdom


In [31]:
# Encontrar a linha com a maior quantidade
df_clean.loc[
    df_clean["Quantity"].idxmax()
]

InvoiceNo                           581483
StockCode                            23843
Description    PAPER CRAFT , LITTLE BIRDIE
Quantity                             80995
InvoiceDate            2011-12-09 09:15:00
UnitPrice                             2.08
CustomerID                         16446.0
Country                     United Kingdom
Name: 540421, dtype: object

In [32]:
# Identificar faturas de cancelamento
eh_cancelamento = (
    df_clean["InvoiceNo"].astype(str).str.startswith("C")
)

# Separar clientes com cancelamentos
clientes_cancelamento = set(
    df_clean.loc[eh_cancelamento, "CustomerID"].unique()
)

# Separar clientes com compras positivas e preço positivo
clientes_compra = set(
    df_clean.loc[
        (~eh_cancelamento)
        & (df_clean["Quantity"] > 0)
        & (df_clean["UnitPrice"] > 0),
        "CustomerID"
    ].unique()
)

# Encontrar clientes que aparecem nos dois grupos
clientes_ambos = clientes_cancelamento & clientes_compra

print("Clientes com cancelamentos:", len(clientes_cancelamento))
print("Clientes com compras:", len(clientes_compra))
print("Clientes com compras e cancelamentos:", len(clientes_ambos))

Clientes com cancelamentos: 1589
Clientes com compras: 4338
Clientes com compras e cancelamentos: 1556


In [33]:
# Filtrar compras válidas preliminarmente
df_compras = df_clean[
    (~df_clean["InvoiceNo"].astype(str).str.startswith("C"))
    & (df_clean["Quantity"] > 0)
    & (df_clean["UnitPrice"] > 0)
].copy()

print("Total de linhas de compras:", len(df_compras))

print("Clientes distintos:", df_compras["CustomerID"].nunique())

print("Faturas distintas:", df_compras["InvoiceNo"].nunique())

Total de linhas de compras: 397884
Clientes distintos: 4338
Faturas distintas: 18532


In [34]:
print("Clientes ausentes:", df_compras["CustomerID"].isna().sum())

print("Quantidades negativas:", (df_compras["Quantity"] < 0).sum())

print("Preços zerados:", (df_compras["UnitPrice"] == 0).sum())

print("Preços negativos:", (df_compras["UnitPrice"] < 0).sum())

print("Faturas iniciadas com C:",
      df_compras["InvoiceNo"].astype(str).str.startswith("C").sum())

Clientes ausentes: 0
Quantidades negativas: 0
Preços zerados: 0
Preços negativos: 0
Faturas iniciadas com C: 0


In [35]:
# Criar a base de trabalho para o RFM
df_rfm = df_compras.copy()

# Conferir as dimensões
print(df_rfm.shape)

(397884, 8)


In [36]:
df_clean = df.dropna(subset=["CustomerID"]).copy()

duplicados = df_clean[df_clean.duplicated(keep=False)]

print(f"Total de registros envolvidos em duplicidade: {len(duplicados)}")
print(f"Total de grupos de duplicidade: {duplicados.groupby(list(df_clean.columns)).ngroups}")

Total de registros envolvidos em duplicidade: 10062
Total de grupos de duplicidade: 4837


In [37]:
duplicados["eh_cancelamento"] = (
    duplicados["InvoiceNo"].astype(str).str.startswith("C")
)

duplicados["eh_cancelamento"].value_counts()

eh_cancelamento
False    10001
True        61
Name: count, dtype: int64

In [38]:
duplicados_compras = df_compras[
    df_compras.duplicated(keep=False)
].copy()

print(f"Registros envolvidos em duplicidade: {len(duplicados_compras)}")
print(
    f"Grupos de duplicidade: "
    f"{duplicados_compras.groupby(list(df_compras.columns)).ngroups}"
)

Registros envolvidos em duplicidade: 10001
Grupos de duplicidade: 4809


In [39]:
frequencia_duplicados = (
    duplicados_compras
    .groupby(list(df_compras.columns))
    .size()
    .value_counts()
    .sort_index()
)

frequencia_duplicados

2     4523
3      234
4       38
5        5
6        6
8        1
12       1
20       1
Name: count, dtype: int64

In [40]:
valor_duplicados = (
    duplicados_compras["Quantity"] *
    duplicados_compras["UnitPrice"]
).sum()

valor_compras = (
    df_compras["Quantity"] *
    df_compras["UnitPrice"]
).sum()

print(f"Valor total dos registros duplicados: £{valor_duplicados:,.2f}")
print(f"Valor total das compras: £{valor_compras:,.2f}")
print(
    f"Percentual do valor representado pelos duplicados: "
    f"{(valor_duplicados / valor_compras) * 100:.2f}%"
)

Valor total dos registros duplicados: £46,640.85
Valor total das compras: £8,911,407.90
Percentual do valor representado pelos duplicados: 0.52%


In [41]:
df_compras_sem_duplicados = df_compras.drop_duplicates().copy()

print(f"Registros antes: {len(df_compras)}")
print(f"Registros depois: {len(df_compras_sem_duplicados)}")
print(f"Registros removidos: {len(df_compras) - len(df_compras_sem_duplicados)}")

Registros antes: 397884
Registros depois: 392692
Registros removidos: 5192


In [42]:
duplicados_excedentes = df_compras[df_compras.duplicated(keep="first")]

valor_removido = (
    duplicados_excedentes["Quantity"] * duplicados_excedentes["UnitPrice"]
).sum()

valor_total_original = (
    df_compras["Quantity"] * df_compras["UnitPrice"]
).sum()

print("Duplicatas restantes:", df_compras_sem_duplicados.duplicated().sum())
print(f"Valor efetivamente removido: £{valor_removido:,.2f}")
print(f"Impacto no valor total: {(valor_removido / valor_total_original) * 100:.2f}%")

Duplicatas restantes: 0
Valor efetivamente removido: £24,199.01
Impacto no valor total: 0.27%


In [43]:
df_clean  = df_compras_sem_duplicados.copy()

df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [44]:
print("Clientes ausentes:", df_clean["CustomerID"].isna().sum())

print("Quantidades negativas:", (df_clean["Quantity"] < 0).sum())

print("Preços zerados:", (df_clean["UnitPrice"] == 0).sum())

print("Preços negativos:", (df_clean["UnitPrice"] < 0).sum())

print("Faturas iniciadas com C:",
      df_clean["InvoiceNo"].astype(str).str.startswith("C").sum())

print("Duplicatas restantes:", df_clean.duplicated().sum())

Clientes ausentes: 0
Quantidades negativas: 0
Preços zerados: 0
Preços negativos: 0
Faturas iniciadas com C: 0
Duplicatas restantes: 0


In [45]:
df_clean["TotalPrice"] = (
    df_clean["Quantity"] *
    df_clean["UnitPrice"]
)

df_clean.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalPrice
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34


In [46]:
# Cria a pasta caso ela ainda não exista
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Salva a base tratada
df_clean.to_csv(
    "../data/processed/customer_transactions_clean.csv",
    index=False,
    encoding="utf-8"
)

In [47]:
df_check = pd.read_csv(
    "../data/processed/customer_transactions_clean.csv"
)

print("Dimensões:", df_check.shape)
print("Duplicatas:", df_check.duplicated().sum())
print("Colunas:", df_check.columns.tolist())

Dimensões: (392692, 9)
Duplicatas: 0
Colunas: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TotalPrice']
